In [0]:
%python

## IMPORTS AND CONFIGURATION
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, sum as _sum, avg, count, countDistinct, round, coalesce, lit
)

'''
 1. Load Gold Layer tables
'''
fact_tx = spark.table("banking_gold.fact_transactions")
fact_tx.printSchema()
dim_cust = spark.table("banking_gold.dim_customers")

dim_acc = spark.table("banking_gold.dim_accounts")
dim_acc.printSchema()
dim_cust.printSchema()
dim_date = spark.table("banking_gold.dim_date")
dim_date.printSchema()

STATISTIC 1: Product Usage by Legal Age Status (is_legal)

In [0]:
# Join dimensions with Fact Table

kpi_age_product_usage = (
    fact_tx
    .join(dim_cust, "customer_sk")
    .join(dim_acc, "account_sk")
    .groupBy(
        col("is_legal").alias("is_legal_age"), 
        col("account_type")
    )
    .agg(
        countDistinct(fact_tx["customer_sk"]).alias("total_unique_customers"),
        count(col("transaction_id")).alias("total_transactions"), 
        round(_sum(col("amount")), 2).alias("total_volume_euros"), 
        round(avg(col("amount")), 2).alias("average_ticket_euros")
    )
    .orderBy("is_legal_age", "account_type")
)

print("=== 1. Product Usage: Legal Age vs Minors ===")
display(kpi_age_product_usage)

# Save report table to Gold layer for consumption
(
    kpi_age_product_usage.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.rpt_age_product_usage")
)

STATISTIC 2: Credit Risk Distribution (Bureau) vs Transaction Volume

In [0]:
kpi_credit_risk = (
    fact_tx
    .join(dim_acc, "account_sk")
    .join(dim_cust, "customer_sk")
    .groupBy(col("risk_grade"), "credit_score")
    .agg(
        countDistinct(fact_tx["customer_sk"]).alias("at_risk_customers"),
        count("transaction_id").alias("total_transactions"),
        round(_sum("amount"), 2).alias("total_volume_eur")
    )
    .orderBy(col("total_volume_eur").desc())
)

print("=== 2. Credit Risk Analysis ===")
display(kpi_credit_risk)

# Write data to gold layer as a report table for consumption
(
    kpi_credit_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.rpt_credit_risk")
)

STATISTIC 3: Operational Performance by Branch Region

In [0]:
kpi_branch_performance = (
    fact_tx
    .join(dim_acc, "account_sk")
    .join(dim_cust, "customer_sk")
    .groupBy(col("branch_city"), col("branch_region"))
    .agg(
        countDistinct(col("account_sk")).alias("active_accounts"),
        count(col("transaction_id")).alias("total_trans"), 
        _sum(col("amount")).alias("total_volume_eur"), 
        round(avg(col("amount")), 2).alias("average_ticket_eur")
    )
    .orderBy(col("active_accounts"), col("total_trans"))
)

print("=== 3. Performance by Region and Branch ===")
display(kpi_branch_performance)

# Save report table to Gold layer for consumption
(
    kpi_branch_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.rpt_branch_performance")
)

STATISTIC 4: Temporal Behavior (Weekend vs Business Day)

In [0]:
kpi_weekend_analysis = (
    fact_tx
    .join(dim_date, col("date_id") == col("transaction_date_id"))
    .groupBy(col("transaction_type"), col("is_weekend"))
    .agg(
        countDistinct(col("transaction_id")).alias("total_trans"),
        _sum(col("amount")).alias("total_volume_eur"), 
        round(avg(col("amount")), 2).alias("average_ticket_eur")
    )
    .orderBy(col("total_trans").asc(), col("total_volume_eur").asc())
)

print("=== 4. Transaction Patterns: Weekend vs Business Day ===")
display(kpi_weekend_analysis)

# Save report table to Gold layer for consumption
(
    kpi_weekend_analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.rpt_weekend_analysis")
)

In [0]:
'''
CHECK THAT REPORTS HAVE BEEN LOADED INTO THE GOLD LAYER
'''

display(spark.table("banking_gold.rpt_age_product_usage"))
display(spark.table("banking_gold.rpt_branch_performance"))
display(spark.table("banking_gold.rpt_credit_risk"))
display(spark.table("banking_gold.rpt_weekend_analysis"))